In [2]:
import polars as pl                            # Import required libraries.

In [3]:
sales = pl.read_csv(r"C:\Users\LENOVO\Desktop\Data Analitika\E-Commerce-Sales-Customer-Analytics\data\raw\Sales.csv")
# Read the first dataset.

In [4]:
sales.glimpse()

Rows: 62884
Columns: 9
$ Order Number  <i64> 366000, 366001, 366001, 366002, 366002, 366002, 366004, 366004, 366005, 366007
$ Line Item     <i64> 1, 1, 2, 1, 2, 3, 1, 2, 1, 1
$ Order Date    <str> '1/1/2016', '1/1/2016', '1/1/2016', '1/1/2016', '1/1/2016', '1/1/2016', '1/1/2016', '1/1/2016', '1/1/2016', '1/1/2016'
$ Delivery Date <str> null, '1/13/2016', '1/13/2016', '1/12/2016', '1/12/2016', '1/12/2016', null, null, null, null
$ CustomerKey   <i64> 265598, 1269051, 1269051, 266019, 266019, 266019, 1107461, 1107461, 844003, 2035771
$ StoreKey      <i64> 10, 0, 0, 0, 0, 0, 38, 38, 33, 43
$ ProductKey    <i64> 1304, 1048, 2007, 1106, 373, 1080, 163, 1529, 421, 1617
$ Quantity      <i64> 1, 2, 1, 7, 1, 4, 6, 2, 4, 1
$ Currency Code <str> 'CAD', 'USD', 'USD', 'CAD', 'CAD', 'CAD', 'GBP', 'GBP', 'EUR', 'USD'



In [5]:
total_null = sum(sales.null_count())[0]
total_rows = len(sales)
null_percentage = round(total_null/total_rows * 100,2)
print(f"Null percentage: {null_percentage} %")
# Only Delivery Date column has null values, which makes up approximately 80 % of total values.

Null percentage: 79.06 %


In [6]:
sales.is_duplicated().sum()
# No duplicate rows were found in the dataset.

0

In [7]:
sales.head(20)

Order Number,Line Item,Order Date,Delivery Date,CustomerKey,StoreKey,ProductKey,Quantity,Currency Code
i64,i64,str,str,i64,i64,i64,i64,str
366000,1,"""1/1/2016""",null,265598,10,1304,1,"""CAD"""
366001,1,"""1/1/2016""","""1/13/2016""",1269051,0,1048,2,"""USD"""
366001,2,"""1/1/2016""","""1/13/2016""",1269051,0,2007,1,"""USD"""
366002,1,"""1/1/2016""","""1/12/2016""",266019,0,1106,7,"""CAD"""
366002,2,"""1/1/2016""","""1/12/2016""",266019,0,373,1,"""CAD"""
…,…,…,…,…,…,…,…,…
366010,1,"""1/1/2016""","""1/8/2016""",370077,0,618,5,"""CAD"""
366011,1,"""1/1/2016""",null,1984985,66,128,7,"""USD"""
366011,2,"""1/1/2016""",null,1984985,66,1638,1,"""USD"""


In [ ]:
sales = sales.with_columns(
    pl.col("Order Date").str.to_date("%m/%d/%Y").alias("Order_Date"),
    pl.col("Delivery Date").str.to_date("%m/%d/%Y").alias("Delivery_Date")
)
sales
# Changed the type of Order Date/Delivery Date columns from str to date.

In [ ]:
sales = sales.drop("Order Date","Delivery Date") 
# Remove unnecessary columns.

In [ ]:
sales = sales.rename({"CustomerKey" : "Customer Key",
                      "StoreKey" : "Store Key",
                      "ProductKey" : "Product Key",
                      "Order_Date" : "Order Date",
                      "Delivery_Date" : "Delivery Date"})
# Standardize column names.

In [ ]:
stores = pl.read_csv(r"C:\Users\LENOVO\Desktop\Data Analitika\E-Commerce-Sales-Customer-Analytics\data\raw\Stores.csv")
stores
# Read the second dataset.

In [ ]:
stores.glimpse()

In [ ]:
stores.null_count()

In [ ]:
stores.is_duplicated().sum()
# No duplicate rows were found in the dataset.

In [ ]:
stores = stores.with_columns(
    pl.col("Open Date").str.to_date("%m/%d/%Y").alias("Open_Date")
)
# Changed the data type of Order Date column to date.

In [ ]:
stores = stores.drop("Open Date")

In [ ]:
stores.rename({"Open_Date" : "Open Date","StoreKey" : "Store Key"})

In [ ]:
stores = stores.filter(
    pl.col("StoreKey") != 0
)
# Dropped the row where square meters is null and Country and State values are "Online" which seems meaningless and does not give 
# any important information.

In [17]:
products = pl.read_csv(r"C:\Users\LENOVO\Desktop\Data Analitika\E-Commerce-Sales-Customer-Analytics\data\raw\Products.csv")
# Read the third dataset

In [18]:
products.glimpse()

Rows: 2517
Columns: 10
$ ProductKey     <i64> 1, 2, 3, 4, 5, 6, 7, 8, 9, 10
$ Product Name   <str> 'Contoso 512MB MP3 Player E51 Silver', 'Contoso 512MB MP3 Player E51 Blue', 'Contoso 1G MP3 Player E100 White', 'Contoso 2G MP3 Player E200 Silver', 'Contoso 2G MP3 Player E200 Red', 'Contoso 2G MP3 Player E200 Black', 'Contoso 2G MP3 Player E200 Blue', 'Contoso 4G MP3 Player E400 Silver', 'Contoso 4G MP3 Player E400 Black', 'Contoso 4G MP3 Player E400 Green'
$ Brand          <str> 'Contoso', 'Contoso', 'Contoso', 'Contoso', 'Contoso', 'Contoso', 'Contoso', 'Contoso', 'Contoso', 'Contoso'
$ Color          <str> 'Silver', 'Blue', 'White', 'Silver', 'Red', 'Black', 'Blue', 'Silver', 'Black', 'Green'
$ Unit Cost USD  <str> '$6.62 ', '$6.62 ', '$7.40 ', '$11.00 ', '$11.00 ', '$11.00 ', '$11.00 ', '$30.58 ', '$30.58 ', '$30.58 '
$ Unit Price USD <str> '$12.99 ', '$12.99 ', '$14.52 ', '$21.57 ', '$21.57 ', '$21.57 ', '$21.57 ', '$59.99 ', '$59.99 ', '$59.99 '
$ SubcategoryKey <i64> 101, 101, 10

In [19]:
products.null_count()
# There are no null values in the dataset.

ProductKey,Product Name,Brand,Color,Unit Cost USD,Unit Price USD,SubcategoryKey,Subcategory,CategoryKey,Category
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,0,0,0,0,0,0,0


In [20]:
products.is_duplicated().sum()
# No duplicate rows were found in the dataset.

0

In [ ]:
products = products.rename(
    {"ProductKey" : "Product Key",
    "SubcategoryKey" : "Subcategory Key",
    "CategoryKey" : "Category Key"}
)
# Standardize column names.

In [ ]:
products

In [21]:
exchange_rates = pl.read_csv(r"C:\Users\LENOVO\Desktop\Data Analitika\E-Commerce-Sales-Customer-Analytics\data\raw\Exchange_Rates.csv")

In [22]:
exchange_rates.null_count()
# There are no null values in the dataset.

Date,Currency,Exchange
u32,u32,u32
0,0,0


In [23]:
exchange_rates.is_duplicated().sum()

0

In [24]:
exchange_rates

Date,Currency,Exchange
str,str,f64
"""1/1/2015""","""USD""",1.0
"""1/1/2015""","""CAD""",1.1583
"""1/1/2015""","""AUD""",1.2214
"""1/1/2015""","""EUR""",0.8237
"""1/1/2015""","""GBP""",0.6415
…,…,…
"""2/20/2021""","""USD""",1.0
"""2/20/2021""","""CAD""",1.261
"""2/20/2021""","""AUD""",1.2723


In [25]:
exchange_rates = exchange_rates.with_columns(
    pl.col("Date").str.to_date("%m/%d/%Y").alias("Date_new")
)

In [26]:
exchange_rates = exchange_rates.drop("Date")
# Remove the previous Date column whose data type was str.

In [27]:
exchange_rates.rename(
    {"Date_new" : "Date"}
)

Currency,Exchange,Date
str,f64,date
"""USD""",1.0,2015-01-01
"""CAD""",1.1583,2015-01-01
"""AUD""",1.2214,2015-01-01
"""EUR""",0.8237,2015-01-01
"""GBP""",0.6415,2015-01-01
…,…,…
"""USD""",1.0,2021-02-20
"""CAD""",1.261,2021-02-20
"""AUD""",1.2723,2021-02-20
